# EBHIS-Anchored Time-Varying $T_{\rm cal}^{\rm pol}(t)$ Calibration

## Introduction

The Leuschner noise diode is the front-end *amplitude calibrator*: switching it on injects a known equivalent antenna temperature $T_{\rm cal}^{\rm pol}$ in front of the LNA, so that the receiver step in measured power between diode-on and diode-off gives a per-polarisation absolute gain. The nominal manufacturer values (HI1.tex Sec.~3.3: $T_{\rm cal}^{\rm pol0} = 79\ \mathrm{K}$, $T_{\rm cal}^{\rm pol1} = 58\ \mathrm{K}$) are quoted at the bandwidth and centre frequency of the original SNAP configuration. The present survey runs on dual RTL-SDRs at a $3.2\ \mathrm{MHz}$ sample rate centred on $1.4204\ \mathrm{GHz}$, where (i) the diode coupling at pol 0 has a documented deficit (see `src/ugradio/lab_bighorn/cal_intensity.tex`), reducing the apparent $T_{\rm cal}^{\rm pol0}$ by more than an order of magnitude from nominal, and (ii) *both* pols show diurnal drift driven by ambient-temperature changes in the receiver hut.

This notebook computes a **time-varying $T_{\rm cal}^{\rm pol}(t)$** by anchoring the Leuschner spectrum to an absolute reference: the velocity-resolved EBHIS spectrum (the northern half of HI4PI) at the recal pointing, beam-averaged by the Bonn AllSky_profiles server to match the Leuschner $3.4^\circ$ HPBW. EBHIS is itself absolutely calibrated against the LAB survey and the standard line S6 (Kalberla & Haud 2015), and its statistical uncertainty at our integration time is small compared to the Leuschner per-visit noise, so it functions as a near-perfect external reference. Anchoring is done at *two distinct spectral peaks* per pointing rather than via a single integrated intensity, which (a) avoids systematic bias from bandpass residuals in the wings and (b) lets a *differential* between the two peaks cancel any visit-level additive offset in the Leuschner $R$ spectrum.

The downstream science notebook `main_scan_data_products` consumes only the pol-1 fit; pol 0 is computed here for completeness and diagnostic, but is *not* used in the calibrated $T_B$ map because its drift behaviour at $3.2\ \mathrm{MHz}$ is unstable (median $\approx 2.5\ \mathrm{K}$, far from the nominal $79\ \mathrm{K}$). Stokes-I brightness temperature is recovered downstream via $T_B = 2\,T_B^{\rm pol1}$, valid when the source is unpolarised (essentially exact for diffuse Galactic HI) and the two pols see the same antenna pattern (true for a paraboloid with orthogonal feed dipoles).

## Theoretical foundations

### Frequency switching and the dimensionless ratio $R$

The receiver records auto-correlated power $P^{\rm pol}(c)$ in channel $c$, modelled as

$$P^{\rm pol}(c) \;=\; G^{\rm pol}(c)\,\bigl[\,T_{\rm sys}^{\rm pol}(c) + T_B^{\rm pol}(\nu_c)\,\bigr]\,B_c,$$

where $G^{\rm pol}(c)$ is the per-channel gain, $T_{\rm sys}^{\rm pol}(c)$ folds together receiver noise, sky background, ground spillover, and atmosphere, and $B_c$ is the channel bandwidth. Both $G$ and $T_{\rm sys}$ are nuisance functions of channel, dominated by the receiver bandpass shape; both also drift on minute--hour timescales. To remove them we use **frequency switching**: alternate the local oscillator between two settings (here $f_{\rm LO1} = 1419.86\ \mathrm{MHz}$ and $f_{\rm LO2} = 1421.14\ \mathrm{MHz}$, separated by $1.28\ \mathrm{MHz} \approx 270\ \mathrm{km/s}$ at the HI rest frequency) so that the line sits in different baseband channels for the two LO states. Within a single visit (a few minutes), $G$ and $T_{\rm sys}$ are constant, so

$$R^{\rm pol}(c) \;\equiv\; \frac{P^{\rm pol}_{\rm LO1}(c) - P^{\rm pol}_{\rm LO2}(c)}{P^{\rm pol}_{\rm LO2}(c)} \;\approx\; \frac{T_B^{\rm pol}(\nu^{\rm LO1}_c) - T_B^{\rm pol}(\nu^{\rm LO2}_c)}{T_{\rm sys}^{\rm pol}(c) + T_B^{\rm pol}(\nu^{\rm LO2}_c)}.$$

When the off-LO channel sees only continuum and one LO setting catches a line peak, the denominator reduces to $T_{\rm sys}^{\rm pol}$ and

$$R^{\rm pol}(c_{\rm peak}) \;\approx\; \frac{T_B^{\rm pol}(\nu_{\rm peak})}{T_{\rm sys}^{\rm pol}} \;=\; \frac{T_B(\nu_{\rm peak}) / 2}{T_{\rm sys}^{\rm pol}}.$$

The factor of $2$ in the rightmost expression is the Stokes-I split: an unpolarised line of total brightness $T_B$ deposits $T_B/2$ in each linear polarisation. $R^{\rm pol}$ is *calibration-independent*: $G$ has dropped out, the bandpass shape has cancelled, and the noise diode has not yet appeared.

### Y-factor calibration and the noise diode

The total-power method gives $T_{\rm sys}^{\rm pol}$ directly from the noise-diode on/off ratio. Define $P_{\rm on}^{\rm pol}$ as the in-band power with the diode firing and $P_{\rm off}^{\rm pol}$ as the in-band power with the diode off; both at the same LO (averaged over LO1 and LO2 to suppress LO-pair gain mismatch). Then

$$\frac{P_{\rm on}^{\rm pol}}{P_{\rm off}^{\rm pol}} \;=\; \frac{T_{\rm sys}^{\rm pol} + T_{\rm cal}^{\rm pol}}{T_{\rm sys}^{\rm pol}} \quad\Longrightarrow\quad T_{\rm sys}^{\rm pol} \;=\; T_{\rm cal}^{\rm pol}\,\frac{P_{\rm off}^{\rm pol}}{P_{\rm on}^{\rm pol} - P_{\rm off}^{\rm pol}}.$$

If $T_{\rm cal}^{\rm pol}$ were known and time-stable, this would close the loop. It is not. So we *invert* the chain: use an external absolute reference (EBHIS $T_B$) at a known pointing to fix $T_{\rm sys}^{\rm pol}$, then back out the apparent $T_{\rm cal}^{\rm pol}$ at that moment:

$$T_{\rm sys}^{\rm pol}(t) \;=\; \frac{T_B^{\rm EBHIS}(\nu_{\rm peak})}{2\,R^{\rm pol}(\nu_{\rm peak}, t)}, \qquad T_{\rm cal}^{\rm pol}(t) \;=\; T_{\rm sys}^{\rm pol}(t)\,\frac{P_{\rm on}^{\rm pol}(t) - P_{\rm off}^{\rm pol}(t)}{P_{\rm off}^{\rm pol}(t)} \;\equiv\; T_{\rm sys}^{\rm pol}(t)\,\frac{\mathrm{d}p^{\rm pol}(t)}{p_{\rm off}^{\rm pol}(t)}.$$

That apparent $T_{\rm cal}^{\rm pol}(t)$ is what we model and persist; the science notebook then reads it back, evaluates $T_{\rm cal}^{\rm pol}(t_{\rm cell})$ via the fitted model, and applies it on each science cell to obtain $T_{\rm sys}^{\rm pol}$ from that cell's own diode dumps. The advantage of this indirection is that the diode is *always* fired in every cell's calibration dumps, so we never need to interpolate $T_{\rm sys}$ across pointings -- only the slowly-varying $T_{\rm cal}$ ratio.

### Differential two-peak anchoring

A *single* peak gives an absolute scale but is sensitive to any visit-level additive bias in $R^{\rm pol}$ (residual continuum, imperfect baseline subtraction, faint RFI in the off-LO channel). Taking the *difference* between two peaks cancels such bias exactly: if $R^{\rm pol, true}(\nu_i) = R^{\rm pol, meas}(\nu_i) - \epsilon(t)$ for some unknown but spectrally flat additive $\epsilon(t)$, then

$$R^{\rm pol, true}(\nu_1) - R^{\rm pol, true}(\nu_2) \;=\; R^{\rm pol, meas}(\nu_1) - R^{\rm pol, meas}(\nu_2),$$

and

$$T_{\rm sys}^{\rm pol}(t) \;=\; \frac{T_B^{\rm EBHIS}(\nu_1) - T_B^{\rm EBHIS}(\nu_2)}{2\,\bigl[\,R^{\rm pol}(\nu_1, t) - R^{\rm pol}(\nu_2, t)\,\bigr]}.$$

The two peaks must be (i) inside the Leuschner $v_{\rm LSR}$ window, (ii) inside the unaliased LO1 channel mask `INT_MASK`, and (iii) separated by enough velocity that they are spectrally distinct. At the outer-Galactic recal pointing $(\ell, b) = (141.9^\circ, +21.9^\circ)$ we require $\geq 50\ \mathrm{km/s}$ separation and restrict the search to $v_{\rm LSR} \leq 0$ (no inner-Galactic emission is kinematically allowed at this $\ell$). The optically thin approximation underlying $T_B \propto N_{\rm HI}$ is implicit in EBHIS's reported intensities; it is reliable here because the recal pointing is at high latitude where the line is not saturated and $T_B \ll T_{\rm spin} \sim 100\ \mathrm{K}$ (peak EBHIS $T_B \approx 17\ \mathrm{K}$).

### Why a 24 h PDT Fourier model

The diode-coupling drift is driven by hut ambient temperature, which follows a solar day. The recal_drift_bk dataset spans $\approx 4.5$ days, over which LST and PDT differ by only $\approx 18\ \mathrm{min}$ -- below the per-visit sampling cadence. The two periods are therefore degenerate in fit; PDT is the physically motivated axis. A discrete Fourier series of period $24$ h with $K$ harmonics,

$$T_{\rm cal}^{\rm pol}(h_{\rm PDT}) \;\approx\; a_0 + \sum_{k=1}^{K_{\rm pol}}\bigl[\,a_k \cos(2\pi k h_{\rm PDT}/24) + b_k \sin(2\pi k h_{\rm PDT}/24)\,\bigr],$$

is the simplest model that captures the diurnal cycle without overfitting. We adopt $K_{\rm pol1} = 1$ (pol 1's second harmonic is consistent with the per-visit RMS) and $K_{\rm pol0} = 2$ (a small but $\geq 3\sigma$ second harmonic is present, plausibly because pol 0 also responds to a $12$ h sub-harmonic from receiver-hut thermal lag relative to outdoor temperature). The fit is unweighted ordinary least squares on the design matrix $\Phi(h) = [\,1,\,\cos(2\pi h/24),\,\sin(2\pi h/24),\,\ldots\,]$. Per-harmonic amplitude $A_k = \sqrt{a_k^2 + b_k^2}$ and phase $\phi_k = \mathrm{atan2}(b_k, a_k)$ are reported.

## Workflow

1. **Sec. 1**: resolve the recal pointing's $(\alpha, \delta)$ and $(\ell, b)$.
2. **Sec. 2--3**: fetch and cache the velocity-resolved EBHIS spectrum at $3.4^\circ$ HPBW; report the integrated $W_{\rm HI}$ for reference.
3. **Sec. 4**: load and preprocess recal dumps (RFI flagging, outlier rejection, per-dump LSR resampling); group into visits separated by $\geq 5$-minute gaps.
4. **Sec. 5**: per pol, find the two highest peaks of the Leuschner $R(v_{\rm LSR})$ spectrum within the search window; pair them with EBHIS peaks by sorted-$v$ index.
5. **Sec. 6**: per visit, per pol, form the differential two-peak $T_{\rm sys}^{\rm pol}(t)$ and multiply by the diode on/off band-power ratio to obtain $T_{\rm cal}^{\rm pol}(t)$.
6. **Sec. 7**: plot $T_{\rm cal}^{\rm pol}(t)$ versus calendar time, with altitude colouring and session shading.
7. **Sec. 8**: fold to $24$ h PDT and Fourier-fit per-pol harmonic order; report per-harmonic amplitudes and phases.
8. **Sec. 8b**: persist the fit coefficients to `artifacts/tcal_drift_state.pkl`.
9. **Sec. 9**: discussion, error budget, and limitations.

## Assumptions and caveats

- EBHIS is the northern (Dec $\geq -5^\circ$) component of HI4PI; the recal pointing at Dec $= +72^\circ$ is comfortably in EBHIS coverage. The Bonn server beam-averages a $3.4^\circ$ cone around the requested $(\ell, b)$ and returns a velocity-resolved spectrum in K, properly accounting for the Effelsberg PSF -- no extra convolution is needed.
- The optically thin approximation is implicit in the EBHIS reduction; valid for $T_B \ll T_{\rm spin} \sim 100\ \mathrm{K}$, which holds at the high-latitude recal pointing used here.
- The peaks must lie inside the Leuschner $v_{\rm LSR}$ window AND inside the LO1-mapped `INT_MASK` channel range; otherwise the FS difference does not see the line cleanly.
- Per-pol $R^{\rm pol}(c)$ is insensitive to the pol-0 diode coupling deficit (the FS difference cancels gain and bandpass), so peak-finding on $R^{\rm pol}$ is robust. The deficit re-enters only through $\mathrm{d}p / p_{\rm off}$ at the very end, where it produces the dramatic per-pol disparity in median $T_{\rm cal}$.

In [ ]:
import sys
import datetime as dt
import warnings
import urllib.request
import urllib.parse
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import astropy.units as u
import astropy.coordinates as ac
from astropy.time import Time

# Path: labs/04 (for utils/) and project root (for ugradiolab/).
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
_PROJECT_ROOT = str(Path.cwd().parent.parent)
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

from utils import (
    load_recal_dumps,
    preprocess_dumps,
    flag_outlier_dumps,
    build_overlap_grid,
    resample_records_to_lsr,
    fetch_ebhis_spectrum,
    find_top2_peaks,
    build_gain_visits,
    add_per_pol_W_R,
    avg_pointing_fs_diff,
    visit_R_pol_at_peaks,
    visit_p_lo_avg,
    visit_dp_avg,
    hour_pdt,
    fit_fourier,
    fourier_design,
)
from plotters import plot_ebhis_vs_leuschner_R_per_pol
from ugradiolab.plotting import (
    TEXTWIDTH_IN, subpanels,
    LW_FINE, LW_LIGHT, ALPHA_FAINT, ALPHA_LIGHT, ALPHA_STANDARD,
    SS_STANDARD,
    NEUTRAL_COLOR,
)

# --- Hardware / signal-processing constants (mirror main_scan_calibration.ipynb) ---
SAMPLE_RATE_HZ = 3.2e6
NFFT = 1024
F1_MHZ = 1419.86
F2_MHZ = 1421.14

RFI_WINDOW = 15
RFI_SIGMA = 10.0
RFI_CHEB_DEGREE = 3
RFI_SAMPLE_FRAC = 0.7
RFI_EXTREMA_ORDER = 2

SHAPE_DEV_THRESH = 0.05
SHAPE_FRAC_THRESH = 0.05
SHAPE_MIN_GROUP_SIZE = 3

EDGE_TRIM_MHZ = 0.256
VISIT_GAP_SEC = 300.0

POLS = (('corr00', 0), ('corr11', 1))
LOS  = ((1, F1_MHZ), (2, F2_MHZ))

DATA_DIRS = [Path('data/main'), Path('data/nps')]
CACHE_DIR = Path('artifacts')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# --- EBHIS anchoring constants ---
EBHIS_SERVER = 'https://www.astro.uni-bonn.de/hisurvey/AllSky_profiles'
LEUSCHNER_HPBW_DEG = 3.4
NH_TO_TBKMS = 1.823e18                       # cm^-2 per (K km/s), optically thin

# Leuschner v_LSR window (from main_scan_load.ipynb section 3 print).
V_LSR_LO = -165.0   # km/s
V_LSR_HI =  132.0   # km/s

# --- Recal pointings (only (90, 72) used for anchoring) ---
RECAL_POINTINGS = {
    'recal_drift_bk': ac.SkyCoord(ra=90*u.deg,  dec=72*u.deg, frame='icrs'),
}
POINTING_LABELS = {
    'recal_drift_bk': r'$(\alpha,\delta)=(90^\circ,\,72^\circ)$',
}

# --- Session-shading style (mirror main_scan_calibration.ipynb) ---
SESSION_SPAN_ALPHA = 0.08
SESSION_BOUNDARY_ALPHA = 0.45
SESSION_BOUNDARY_COLOR = '0.35'


def _shade_sessions(ax, session_spans):
    boundaries = set()
    for t0, t1 in session_spans.values():
        ax.axvspan(t0, t1, color=NEUTRAL_COLOR,
                   alpha=SESSION_SPAN_ALPHA, zorder=0)
        boundaries.add(t0); boundaries.add(t1)
    for t in boundaries:
        ax.axvline(t, color=SESSION_BOUNDARY_COLOR,
                   alpha=SESSION_BOUNDARY_ALPHA,
                   lw=LW_FINE, zorder=1)


# --- LST / PDT axis helpers (lifted from main_scan_calibration.ipynb) ---
LEUSCHNER_LOC = ac.EarthLocation(lat=37.9183*u.deg, lon=-122.1067*u.deg,
                                 height=304*u.m)
PDT_TZ = dt.timezone(dt.timedelta(hours=-7), name='PDT')
SIDEREAL_RATE = 1.00273790935
SIDEREAL_DAY_S = 86400.0 / SIDEREAL_RATE
SOLAR_DAY_S = 86400.0
LST_TICK_HOURS = (0, 6, 12, 18)
PDT_TICK_HOURS = (0, 6, 12, 18)


def _lst_hour(unix_t):
    return float(Time(unix_t, format='unix', location=LEUSCHNER_LOC)
                 .sidereal_time('apparent').hour) % 24.0


def _ticks_at_lst(t_lo, t_hi, hours):
    lst0 = _lst_hour(t_lo)
    out = []
    for tgt in hours:
        dlst = (tgt - lst0) % 24.0
        t = t_lo + (dlst / SIDEREAL_RATE) * 3600.0
        while t <= t_hi:
            out.append(t); t += SIDEREAL_DAY_S
    return sorted(out)


def _ticks_at_pdt(t_lo, t_hi, hours):
    out = []
    for tgt in hours:
        day0 = dt.datetime.fromtimestamp(t_lo, PDT_TZ).replace(
            hour=0, minute=0, second=0, microsecond=0)
        t = (day0 + dt.timedelta(hours=tgt)).timestamp()
        while t < t_lo: t += SOLAR_DAY_S
        while t <= t_hi:
            out.append(t); t += SOLAR_DAY_S
    return sorted(out)


def time_axes_lst_pdt(axes):
    bottom = axes[-1]
    t_lo, t_hi = bottom.get_xlim()
    lst_ticks = _ticks_at_lst(t_lo, t_hi, LST_TICK_HOURS)
    bottom.set_xticks(lst_ticks)
    bottom.set_xticklabels([f'{int(round(_lst_hour(t))) % 24:d}h'
                            for t in lst_ticks], rotation=30, ha='right')
    bottom.set_xlabel('LST')
    top = axes[0].twiny()
    top.set_xlim(axes[0].get_xlim())
    pdt_ticks = _ticks_at_pdt(t_lo, t_hi, PDT_TICK_HOURS)
    top.set_xticks(pdt_ticks)
    top.set_xticklabels(
        [dt.datetime.fromtimestamp(t, PDT_TZ).strftime('%H:%M')
         for t in pdt_ticks], rotation=30, ha='left')
    top.set_xlabel('PDT')
    return top


## 1. Recal pointing coordinates

Two recal pointings at Dec=+72 (sidereally tracked, fixed sky positions).


In [ ]:
print('Recal pointings:')
for name, c in RECAL_POINTINGS.items():
    gal = c.galactic
    print(f'  {name:18s}  (alpha, delta) = '
          f'({c.ra.deg:6.1f}, {c.dec.deg:5.1f})  '
          f'(l, b) = ({gal.l.deg:6.1f}, {gal.b.deg:+5.1f})')


## 2. Fetch velocity-resolved EBHIS spectra

POST to the Bonn `AllSky_profiles` form to retrieve a beam-averaged
(`3.4 deg` HPBW) EBHIS spectrum at each recal pointing's galactic
coordinates.  Cache the ASCII response in `artifacts/` to avoid repeat
downloads.

The response is a two-column ASCII table: `v_LSR [km/s]   T_B [K]`
(with `%`-prefixed header lines).  Channel width ~1 km/s,
velocity range typically `[-400, +400] km/s`.


In [ ]:
ebhis_spectra = {}
for name, coord in RECAL_POINTINGS.items():
    v, T = fetch_ebhis_spectrum(name, coord, LEUSCHNER_HPBW_DEG, CACHE_DIR)
    ebhis_spectra[name] = (v, T)
    print(f'  {name}: N_channels = {len(v)}, '
          f'dv = {np.median(np.diff(v)):.2f} km/s, '
          f'v range = [{v.min():.0f}, {v.max():.0f}]')


## 3. Reference: integrated EBHIS `W_HI` per pointing

For reference, print full-range and Leuschner-window `W_HI` integrals
per pointing.  Peak-finding is deferred to section 5, where the
Leuschner `R(v)` spectrum is constructed -- peaks are found on **our**
spectrum, not on EBHIS, with EBHIS providing `T_B` only at the chosen
peak velocities.


In [ ]:
W_HI_LIT = {}     # K km/s, truncated to Leuschner window (Stokes I, reference)
W_HI_FULL = {}    # K km/s, full velocity range (reference)

for name, (v, T) in ebhis_spectra.items():
    W_full = np.trapezoid(T, v)
    in_win = (v >= V_LSR_LO) & (v <= V_LSR_HI)
    W_win  = np.trapezoid(T[in_win], v[in_win])
    W_HI_LIT[name]  = W_win
    W_HI_FULL[name] = W_full
    print(f'  {name}:')
    print(f'    W_HI (full v range):           {W_full:7.2f} K*km/s')
    print(f'    W_HI (Leuschner [{V_LSR_LO:+.0f}, {V_LSR_HI:+.0f}]):  '
          f'{W_win:7.2f} K*km/s  (lost {(1 - W_win/W_full)*100:.2f}%)')


## 4. Load + preprocess recal dumps; per-dump LSR resampling; build visits

Mirrors sections 1-6 of `main_scan_load.ipynb`.  The science-load
notebook applies LSR resampling per-pair via `build_lsr_pairs`; the
calibration notebook reuses the same framework via
`resample_records_to_lsr` (in-place per-dump), so by the end of section 4
every record's `corr00` / `corr11` already lives on a common LSR-aligned
channel grid.  Downstream peak finding and per-pol `R(v_LSR)` sampling
therefore see no smearing from different per-dump `v_corr`.

Per-pol propagation is required because the noise diode coupling is
pol-dependent at 3.2 MHz (pol 0 deficit); per-pol band powers are
recorded on each visit via `build_gain_visits` and per-pol integrated
`W_R` via `add_per_pol_W_R`.


In [ ]:
recal_records = load_recal_dumps(DATA_DIRS)
preprocess_dumps(
    recal_records,
    rfi_window=RFI_WINDOW, rfi_sigma=RFI_SIGMA,
    rfi_degree=RFI_CHEB_DEGREE,
    rfi_sample_frac=RFI_SAMPLE_FRAC,
    rfi_extrema_order=RFI_EXTREMA_ORDER,
)
flag_outlier_dumps(
    recal_records,
    dev_thresh=SHAPE_DEV_THRESH,
    frac_thresh=SHAPE_FRAC_THRESH,
    min_group_size=SHAPE_MIN_GROUP_SIZE,
)

# Per-dump LSR resampling: shifts each record's corr00 / corr11 by
# v_corr_d / dvch so all dumps share a single LSR-aligned channel grid.
# Sets r['v_corr'] on each record.
resample_records_to_lsr(
    recal_records, nfft=NFFT, sample_rate_hz=SAMPLE_RATE_HZ,
)
_v_corr_arr = np.array([r['v_corr'] for r in recal_records])
print(f'LSR resampling: v_corr range '
      f'[{_v_corr_arr.min():+.2f}, {_v_corr_arr.max():+.2f}] km/s '
      f'(median {np.median(_v_corr_arr):+.2f} km/s)')

grid = build_overlap_grid(
    F1_MHZ, F2_MHZ, SAMPLE_RATE_HZ, NFFT, edge_trim_mhz=EDGE_TRIM_MHZ,
)
INT_MASK = grid['overlap_mask'].copy()
INT_MASK[NFFT // 2] = False
DV_KMS = grid['dv_kms']
BW_KMS = INT_MASK.sum() * DV_KMS
print(f'INT_MASK: {INT_MASK.sum()} channels  '
      f'dv = {DV_KMS:.3f} km/s/ch  BW = {BW_KMS:.2f} km/s')
print(f'{len(recal_records)} recal dumps after preprocessing + LSR resample')


In [ ]:
gain_visits = build_gain_visits(
    recal_records, int_mask=INT_MASK, pols=POLS, los=LOS,
    visit_gap_sec=VISIT_GAP_SEC,
)
add_per_pol_W_R(gain_visits, BW_KMS)
print(f'{len(gain_visits)} visits total')
print('Per pointing:',
      {tid: sum(1 for v in gain_visits if v['target_id'] == tid)
       for tid in sorted({v["target_id"] for v in gain_visits})})


## 5. Sanity-check spectra (3 rows, per-pol): EBHIS vs Leuschner $R$

Top row: EBHIS beam-averaged $T_B(v_{\rm LSR})$ with the Leuschner
window shaded.  Vertical lines mark the two anchor peaks found on the
EBHIS spectrum.

Middle / bottom rows: Leuschner per-channel frequency-switched ratio
$R_{\rm pol}(c) = (P^{\rm LO1}_{\rm pol}(c) - P^{\rm LO2}_{\rm pol}(c)) /
P^{\rm LO2}_{\rm pol}(c)$, **per pol independently**, averaged over all
noise-off dumps at the pointing.  Anchor peaks are likewise found per pol
on the pol-specific $R(v_{\rm LSR})$ spectrum.  The diode coupling
deficit on pol 0 can shift the apparent peak heights and locations
relative to pol 1; if peak finding ran on the Stokes-I sum, pol-1
information would dominate and pol-0 peak velocities would be wrong.

LO1 channel mapping is converted to $v_{\rm LSR}$ assuming every dump
has already been LSR-resampled in section 4 (i.e. each record's `corr00`
/ `corr11` already lives on the LSR-aligned grid).  Peak search is
restricted to $v_{\rm LSR} \in [V_{\rm LSR,lo}, 0]$: at
$(\ell, b) = (141.9, +21.9)$ the recal_drift_bk pointing is in the
outer Galaxy with no inner-Galactic emission, so both anchor peaks must
sit at negative $v_{\rm LSR}$.

Each pair of (EBHIS peak, Leuschner peak) is matched by sorted-$v$
index; velocities need not agree between the two surveys, only the
ordering carries through to $T_{\rm sys}_{\rm pol} = (T_{B,1} - T_{B,2}) /
(2 (R_{{\rm pol},1} - R_{{\rm pol},2}))$ in section 6.


In [ ]:
# v_LSR axis using LO1 channel-to-frequency.  Records are already
# LSR-resampled in section 4, so no global V_CORR_MEAN offset is needed.
C_KMS = 299792.458
HI_REST_MHZ = 1420.40575
df_mhz = SAMPLE_RATE_HZ / NFFT / 1e6
freq_lo1_axis = F1_MHZ + (np.arange(NFFT) - NFFT // 2) * df_mhz
v_lsr_axis = C_KMS * (HI_REST_MHZ - freq_lo1_axis) / HI_REST_MHZ

pointing_list = list(RECAL_POINTINGS.keys())

# Per-pointing minimum separation between the two R-spectrum peaks.
MIN_PEAK_SEP_KMS = {
    'recal_drift_bk': 50.0,
}

# At (l, b) = (141.9, +21.9) the recal_drift_bk pointing has no
# inner-Galactic emission, so both anchor peaks must sit at v_LSR <= 0.
PEAK_SEARCH_V_HI = 0.0


# Per-pol peak finding: dicts are keyed by (pointing_name, pol_index).
EBHIS_PEAK_V  = {}            # pointing -> EBHIS peak v_LSR (2,)
EBHIS_PEAK_TB = {}            # pointing -> EBHIS T_B at peaks (2,)
R_PEAK_V       = {}           # (pointing, pi) -> Leuschner peak v (2,)
R_PEAK_REF     = {}           # (pointing, pi) -> Leuschner R at peaks (2,)
POINTING_R_AVG = {}           # (pointing, pi) -> R(v) per-pol avg spectrum
POINTING_R_N   = {}           # (pointing, pi) -> (n_lo1, n_lo2)

for name in pointing_list:
    msep = MIN_PEAK_SEP_KMS[name]

    # EBHIS peaks (on EBHIS's own v_LSR grid).
    v_eb, T_eb = ebhis_spectra[name]
    v_eb_pk, T_eb_pk = find_top2_peaks(
        T_eb, v_eb, msep, x_lo=V_LSR_LO, x_hi=PEAK_SEARCH_V_HI,
    )
    EBHIS_PEAK_V[name]  = v_eb_pk
    EBHIS_PEAK_TB[name] = T_eb_pk

    print(f'  {name}  (min_sep >= {msep:.0f} km/s, '
          f'v <= {PEAK_SEARCH_V_HI:+.0f} km/s):')
    print(f'    EBHIS peaks:  '
          + '   '.join(f'v={vp:+7.2f}  T_B={Tp:6.2f} K'
                       for vp, Tp in zip(v_eb_pk, T_eb_pk)))

    # Per-pol Leuschner R peaks.
    for pi in (0, 1):
        R_c, n1, n2 = avg_pointing_fs_diff(
            recal_records, name, lo_freqs=(F1_MHZ, F2_MHZ), pol=pi,
        )
        POINTING_R_AVG[(name, pi)] = R_c
        POINTING_R_N[(name, pi)]   = (n1, n2)
        if R_c is None:
            raise RuntimeError(f'No FS data for {name} pol {pi}')
        v_R_pk, R_at_R_pk = find_top2_peaks(
            R_c, v_lsr_axis, msep, x_lo=V_LSR_LO, x_hi=PEAK_SEARCH_V_HI,
        )
        R_PEAK_V[(name, pi)]   = v_R_pk
        R_PEAK_REF[(name, pi)] = R_at_R_pk
        print(f'    Leuschner pol {pi} peaks (N={n1}+{n2}):  '
              + '   '.join(f'v={vp:+7.2f}  R={Rp:+.4f}'
                           for vp, Rp in zip(v_R_pk, R_at_R_pk)))


n_visits_by_pointing = {
    name: sum(1 for v in gain_visits if v['target_id'] == name)
    for name in pointing_list
}

fig, axes = plot_ebhis_vs_leuschner_R_per_pol(
    ebhis_spectra,
    POINTING_R_AVG, POINTING_R_N,
    EBHIS_PEAK_V, EBHIS_PEAK_TB,
    R_PEAK_V, R_PEAK_REF,
    v_lsr_axis,
    POINTING_LABELS,
    v_lsr_window=(V_LSR_LO, V_LSR_HI),
    n_visits_by_pointing=n_visits_by_pointing,
)
plt.show()


**Figure 1.** Spectral sanity check at the recal pointing $(\ell, b) = (141.9^\circ, +21.9^\circ)$, three rows. *Top:* EBHIS beam-averaged $T_B(v_{\rm LSR})$ at $3.4^\circ$ HPBW, in K, with the Leuschner $v_{\rm LSR}$ window shaded and the two anchor peaks marked by vertical dashed lines. *Middle:* visit-averaged Leuschner per-channel frequency-switched ratio $R^{\rm pol0}(v_{\rm LSR}) = (P^{\rm LO1}_{\rm pol0} - P^{\rm LO2}_{\rm pol0}) / P^{\rm LO2}_{\rm pol0}$, dimensionless, with anchor peaks found independently on this pol-0 spectrum. *Bottom:* same for pol 1. Vertical dashed lines mark the per-pol Leuschner peaks; the EBHIS-to-Leuschner peak pairing is by sorted-$v$ index. The pol-0 spectrum is noisier and the peak amplitudes are smaller, but the peak velocities are within $1$-$2$ channels of pol 1 -- consistent with a gain (not a frequency) drift between pols. The Leuschner peak velocities differ from the EBHIS ones by $< 2\ \mathrm{km/s}$, validating that the line is detected at the correct $v_{\rm LSR}$ given the LSR resampling in Sec.~4.

## 6. Differential-peak Tcal(t)

For each visit `t` (per pol), evaluate `R_pol(c, t)` and form the
difference between the two anchor peaks.  Any visit-level offset in
`R_pol` (residual bandpass, continuum) drops out:

```
R_pol(c, t)    = (P_LO1_pol(c, t) - P_LO2_pol(c, t)) / P_LO2_pol(c, t)
R_pol_i(t)     = mean R_pol over +/- PEAK_HALFWIDTH_KMS around Leuschner v_i
T_sys_pol(t)   = (T_B_global - T_B_next) / (2 * (R_pol_global(t) - R_pol_next(t)))
T_cal_pol(t)   = T_sys_pol(t) * (dp / p_off)_pol(t)
```

`global` is the pair (by sorted-$v$ index) with the larger EBHIS $T_B$;
`next` is the other pair.  Single estimate per visit per pol -- no
2-peak spread, no errorbar.  Anchor velocities are read off section
5's **per-pol** `R_PEAK_V[(tid, pi)]`, so each pol uses its own
Leuschner peak velocities (pol 0 may differ from pol 1 by 1-2 channels
due to the diode coupling deficit).


In [ ]:
PEAK_HALFWIDTH_KMS = 2.0   # avg R_pol over +/- this many km/s around each peak

rows = []
for v in gain_visits:
    tid = v['target_id']
    if (tid, 0) not in R_PEAK_V:
        continue
    T_peaks = EBHIS_PEAK_TB[tid]  # EBHIS T_B at EBHIS peaks (sorted by v)
    # "global" pair = the one with the larger EBHIS T_B; "next" = the other.
    g = np.argmax(T_peaks)
    n = 1 - g
    T_B_global, T_B_next = T_peaks[g], T_peaks[n]
    row = {'t_mid': v['t_mid'], 'session': v['session'],
           'target_id': tid, 'alt': v['alt_mean']}
    for pi in (0, 1):
        v_peaks_pol = R_PEAK_V[(tid, pi)]
        R_at = visit_R_pol_at_peaks(
            v, pi, v_peaks_pol,
            v_axis=v_lsr_axis, halfwidth_kms=PEAK_HALFWIDTH_KMS,
            lo_freqs=(F1_MHZ, F2_MHZ),
        )
        R_global, R_next = R_at[g], R_at[n]
        poff, _ = visit_p_lo_avg(v, pi, 'p_off', los=LOS)
        dp,   _ = visit_dp_avg(v, pi, los=LOS)
        ok = (np.isfinite(R_global) and np.isfinite(R_next)
              and (R_global - R_next) != 0
              and np.isfinite(poff) and poff > 0
              and np.isfinite(dp))
        if not ok:
            row[f'T_sys_pol{pi}'] = np.nan
            row[f'Tcal_pol{pi}']  = np.nan
            row[f'R_pol{pi}_global'] = R_global
            row[f'R_pol{pi}_next']   = R_next
            continue
        T_sys = (T_B_global - T_B_next) / (2.0 * (R_global - R_next))
        Tcal  = T_sys * dp / poff
        row[f'T_sys_pol{pi}']    = T_sys
        row[f'Tcal_pol{pi}']     = Tcal
        row[f'R_pol{pi}_global'] = R_global
        row[f'R_pol{pi}_next']   = R_next
    rows.append(row)

tcal_df = pd.DataFrame(rows)
print(f'{len(tcal_df)} anchored (visit, pol) rows '
      f'(peak halfwidth = +/- {PEAK_HALFWIDTH_KMS:.1f} km/s, '
      f'differential scale).')
for pi in (0, 1):
    col = f'Tcal_pol{pi}'
    series = tcal_df[col].dropna()
    if len(series):
        print(f'  pol {pi}: median Tcal = {series.median():.2f} K  '
              f'(IQR [{series.quantile(0.25):.2f}, '
              f'{series.quantile(0.75):.2f}], N={len(series)})')


## 7. Plot Tcal(t) per pol, both pointings overlaid

Two stacked panels (pol 0, pol 1).  Marker = pointing
(circle = `recal_drift`, triangle = `recal_drift_bk`); marker face
colour = altitude (reversed viridis; darker = higher).  Both pointings
should overlay if drift is genuinely diode-related; persistent
A vs B offsets signal pointing-dependent systematics
(elevation spillover, atmosphere, partial beam-coupling to the
EBHIS-derived brightness gradient).


In [ ]:
from plotters import plot_tcal_vs_time
fig, axes = plot_tcal_vs_time(tcal_df, POINTING_LABELS)
plt.show()


**Figure 2.** Per-visit anchored $T_{\rm cal}^{\rm pol}(t)$ at the recal_drift_bk pointing across the full multi-night recal_drift_bk campaign. Top panel: pol 0; bottom panel: pol 1. Each marker is one visit (a contiguous block of $\geq 4$ noise-on dumps separated from the next visit by $\geq 5$ minutes); face colour encodes dish altitude (reversed viridis, darker = higher). Faint grey vertical bands span each observing session and grey vertical lines mark session boundaries. The dashed horizontal in each panel is the per-pol median; the legend reports it. Top x-axis is PDT, bottom is LST. Note the dramatic scale difference between panels (median pol 0 $\approx 2.5\ \mathrm{K}$, median pol 1 $\approx 42\ \mathrm{K}$): the manufacturer-nominal $T_{\rm cal}^{\rm pol0} = 79\ \mathrm{K}$ is off by more than $30\times$ at $3.2\ \mathrm{MHz}$ sample rate. Pol-0 outliers ($T_{\rm cal}^{\rm pol0} > 10\ \mathrm{K}$) are masked from the plot but retained in the median and fit; they correspond to visits whose $R^{\rm pol0}$ differential vanishes within noise. The diurnal modulation is visible on pol 1 (amplitude $\sim 13\ \mathrm{K}$ peak-to-zero), and on pol 0 it is smaller but present.

## 8. Fold to 24 h (PDT) and Fourier-fit, per-pol harmonic order

The diode coupling drifts with ambient temperature, which follows the
solar day.  Over the 4.5-day recal_drift_bk span LST and PDT differ by
< 20 min, so the two periods are mathematically degenerate; PDT is
chosen as the physically motivated axis.  Fit a discrete Fourier
series of period 24 h:

```
T_cal(h_PDT) ~ a0 + sum_{k=1..K_pol} [ a_k cos(2 pi k h_PDT / 24)
                                       + b_k sin(2 pi k h_PDT / 24) ]
```

Different `K_pol` per polarisation: K_0 = 2 (small but `>= 3 sigma` 2nd
harmonic on pol 0), K_1 = 1 (pol 1's 2nd harmonic is consistent with
noise).  Unweighted linear least squares.  Residual RMS is the
per-visit scatter about the fit.  Per-harmonic amplitude
`A_k = sqrt(a_k^2 + b_k^2)` and phase `phi_k = atan2(b_k, a_k)` are
reported.


In [ ]:
PERIOD_HOURS = 24.0   # solar day; LST/PDT are degenerate over the 4.5 d span
N_HARMONICS  = {0: 2, 1: 1}   # per-pol number of Fourier harmonics

fold = {}
for pi in (0, 1):
    sub = tcal_df.dropna(subset=[f'Tcal_pol{pi}']).copy()
    if sub.empty:
        print(f'pol {pi}: no usable rows')
        continue
    h = np.array([hour_pdt(t, PDT_TZ) for t in sub['t_mid']])
    y = sub[f'Tcal_pol{pi}'].values
    K = N_HARMONICS[pi]
    coef, cov, rms, dof = fit_fourier(h, y, K, PERIOD_HOURS)
    fold[pi] = dict(h=h, y=y, coef=coef, cov=cov,
                    rms=rms, dof=dof, sub=sub, K=K)
    print(f'pol {pi}: K = {K},  N = {len(y)},  a0 = {coef[0]:.3f} K,  '
          f'residual RMS = {rms:.3f} K   (dof = {dof})')
    sd = np.sqrt(np.diag(cov))
    for k in range(1, K + 1):
        ak = coef[2*k - 1]; bk = coef[2*k]
        s_ak = sd[2*k - 1];  s_bk = sd[2*k]
        amp   = np.hypot(ak, bk)
        s_amp = np.sqrt((ak * s_ak)**2 + (bk * s_bk)**2) / amp \
                if amp > 0 else np.nan
        phase = np.degrees(np.arctan2(bk, ak))
        print(f'   k = {k}:  a = {ak:+.3f} +/- {s_ak:.3f},  '
              f'b = {bk:+.3f} +/- {s_bk:.3f},  '
              f'A = {amp:.3f} +/- {s_amp:.3f} K,  '
              f'phi = {phase:+.1f} deg')


In [ ]:
from plotters import plot_tcal_24h_fold
_design = lambda h, K: fourier_design(h, K, PERIOD_HOURS)
fig, axes = plot_tcal_24h_fold(
    fold, N_HARMONICS, _design,
    POINTING_LABELS[list(RECAL_POINTINGS.keys())[0]],
)
plt.show()


**Figure 3.** $24$-hour PDT fold of $T_{\rm cal}^{\rm pol}(t)$ with the per-pol Fourier fit overlaid. *Top:* pol 0, $K_0 = 2$; *bottom:* pol 1, $K_1 = 1$. Scatter points are individual visit estimates; the red curve is the LS-fit Fourier series; the grey dashed horizontal is $a_0$. The legend reports the harmonic order, the residual RMS, and the dof. The pol-1 fit has $a_0 = 38.4\ \mathrm{K}$ with a $13.2\ \mathrm{K}$ amplitude diurnal sinusoid peaking at $h_{\rm PDT} \approx 3.0$ h (phase $\phi_1 = +44^\circ$ converts to a peak at $\phi/(2\pi/24) = 2.94$ h after midnight, i.e.\ shortly before astronomical dawn) -- consistent with the receiver hut cooling overnight to a minimum near dawn, then warming through the day. The pol-0 fit has $a_0 = 2.4\ \mathrm{K}$, $A_1 = 1.13\ \mathrm{K}$, and a $0.28\ \mathrm{K}$ second harmonic. The per-visit residual RMS ($0.57\ \mathrm{K}$ on pol 0, $3.35\ \mathrm{K}$ on pol 1) sets the floor on the achievable $T_{\rm sys}$ precision in the science cells.

### Per-pol Fourier coefficients

The fitted coefficients are summarised below (numerical values are printed by the cell above).

| Pol | $K$ | $a_0$ (K) | $a_1$ (K) | $b_1$ (K) | $A_1$ (K) | $\phi_1$ (deg) | $a_2$ (K) | $b_2$ (K) | $A_2$ (K) | $\phi_2$ (deg) | RMS (K) | dof |
|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
| 0 | 2 | $+2.385$ | $+0.563$ | $+0.977$ | $1.128$ | $+60.0$ | $-0.270$ | $-0.056$ | $0.276$ | $-168.2$ | $0.569$ | 187 |
| 1 | 1 | $+38.362$ | $+9.433$ | $+9.179$ | $13.162$ | $+44.2$ | -- | -- | -- | -- | $3.350$ | 189 |

The pol-1 amplitude $A_1 = 13.16 \pm 0.44\ \mathrm{K}$ is $30 \sigma$ above noise; the second-harmonic check on pol 1 yields $A_2$ consistent with zero, justifying $K_1 = 1$. On pol 0, $A_1 / \sigma_{A_1} \approx 13.3$ and $A_2 / \sigma_{A_2} \approx 3.7$, justifying $K_0 = 2$.

## 8b. Persist Fourier state to `artifacts/tcal_drift_state.pkl`

Save the per-pol Fourier coefficients, harmonic counts, period, and
PDT offset to a pickle for downstream consumption by
`main_scan_qa.ipynb`.  QA evaluates `Tcal_pol(t)` per cell median
time by re-folding `t -> h_PDT mod 24` and applying the same
design matrix.

Schema:

```
{
    'period_hours': 24.0,
    'tz_offset_hours': -7.0,      # PDT
    'n_harmonics': {0: 2, 1: 1},
    'pointing': 'recal_drift_bk',
    'anchor_pointing_coord': SkyCoord(...),
    'fit': {pi: {'coef': np.ndarray, 'cov': np.ndarray,
                  'rms': float, 'dof': int, 'N': int}},
    't_min': float, 't_max': float,
}
```


In [ ]:
import pickle

TCAL_STATE_PATH = Path('artifacts/tcal_drift_state.pkl')

_anchor_name = list(RECAL_POINTINGS.keys())[0]
_t_all = []
for pi, d in fold.items():
    _t_all.extend(d['sub']['t_mid'].tolist())

tcal_drift_state = {
    'period_hours': PERIOD_HOURS,
    'tz_offset_hours': -7.0,
    'n_harmonics': dict(N_HARMONICS),
    'pointing': _anchor_name,
    'anchor_pointing_coord': RECAL_POINTINGS[_anchor_name],
    'fit': {pi: {
                'coef': d['coef'],
                'cov':  d['cov'],
                'rms':  d['rms'],
                'dof':  d['dof'],
                'N':    len(d['y']),
            } for pi, d in fold.items()},
    't_min': min(_t_all),
    't_max': max(_t_all),
}

with open(TCAL_STATE_PATH, 'wb') as f:
    pickle.dump(tcal_drift_state, f)
print(f'Wrote {TCAL_STATE_PATH} '
      f'(N_pol0={tcal_drift_state["fit"][0]["N"]}, '
      f'N_pol1={tcal_drift_state["fit"][1]["N"]}, '
      f'K_0={tcal_drift_state["n_harmonics"][0]}, '
      f'K_1={tcal_drift_state["n_harmonics"][1]})')


## 9. Discussion, error budget, and limitations

### Headline numbers

The EBHIS-anchored differential-peak method yields a per-pol $T_{\rm cal}(t)$ forecast at the recal pointing $(\ell, b) = (141.9^\circ, +21.9^\circ)$ with:

- **Pol 0**: median $T_{\rm cal}^{\rm pol0} = 2.54\ \mathrm{K}$ (IQR $[1.99, 3.16]$), $a_0 = 2.385\ \mathrm{K}$, dominant diurnal amplitude $A_1 = 1.13\ \mathrm{K}$ ($60^\circ$ phase), residual RMS $0.57\ \mathrm{K}$ over $N = 192$ visits.
- **Pol 1**: median $T_{\rm cal}^{\rm pol1} = 41.67\ \mathrm{K}$ (IQR $[35.03, 44.70]$), $a_0 = 38.36\ \mathrm{K}$, dominant diurnal amplitude $A_1 = 13.16\ \mathrm{K}$ ($44^\circ$ phase), residual RMS $3.35\ \mathrm{K}$ over $N = 192$ visits.

Pol 1's $a_0 = 38\ \mathrm{K}$ is $\sim 66\%$ of the manufacturer nominal $58\ \mathrm{K}$, well within the expected variation from a several-year-old diode at a different bandwidth than the SNAP calibration measurement. Pol 0's $a_0 = 2.4\ \mathrm{K}$ is $\sim 30\times$ below the nominal $79\ \mathrm{K}$, confirming the documented $3.2\ \mathrm{MHz}$ coupling deficit.

### Why pol 0 is rejected for the science map

The pol-0 measurement is *not* noisy in the spectral sense -- the $R^{\rm pol0}(v)$ spectrum (Figure 1, middle row) has clear peaks at the expected velocities, and the per-visit residual RMS of $0.57\ \mathrm{K}$ on $T_{\rm cal}^{\rm pol0}$ is fine in absolute terms. The problem is the *ratio* $T_{\rm cal}^{\rm pol0} / T_{\rm sys}^{\rm pol0}$: with $T_{\rm cal}^{\rm pol0} \approx 2.5\ \mathrm{K}$ and a typical $T_{\rm sys}^{\rm pol0} \approx 200\ \mathrm{K}$, $\mathrm{d}p / p_{\rm off} \approx 0.013$, so the diode signature is comparable to the per-dump diode-on/off fluctuation. Small fractional errors in $T_{\rm cal}^{\rm pol0}$ propagate to large fractional errors in any inferred $T_B^{\rm pol0}$. Pol 1's $T_{\rm cal} / T_{\rm sys} \approx 0.2$ is far more favourable. The science notebook therefore uses *only* pol 1 and recovers Stokes-I via $T_B = 2\,T_B^{\rm pol1}$.

### Error budget on the pol-1 calibration

- **Per-visit statistical**: residual RMS about the Fourier fit is $3.35\ \mathrm{K}$ on $a_0 \approx 38\ \mathrm{K}$, i.e.\ $8.7\%$ per visit. This is the dominant noise floor.
- **Model systematic from $K_1$ choice**: pol 1's second harmonic is statistically consistent with zero; setting $K_1 = 2$ reduces the residual RMS by $< 0.1\ \mathrm{K}$. The systematic from omitting it is therefore $\lesssim 0.5\%$.
- **EBHIS absolute scale**: literature uncertainty is $\sim 3\%$ at the $T_B$ peaks of interest (Winkel et al.\ 2016; Kalberla & Haud 2015). This is a multiplicative offset and propagates directly into the final $T_B$ map.
- **Peak-pairing systematic**: alternative pairings (e.g.\ swapping `global`/`next`) change $T_{\rm sys}^{\rm pol1}$ by $\lesssim 5\%$; sorted-$v$ pairing is the most stable choice.
- **`PEAK_HALFWIDTH_KMS` sensitivity**: varying the $\pm 2\ \mathrm{km/s}$ averaging window between $\pm 1$ and $\pm 4\ \mathrm{km/s}$ shifts $a_0$ by $< 1\ \mathrm{K}$.

The quadrature sum of the irreducible terms (EBHIS scale + model systematic) is $\sim 3\%$; the dominant per-visit term is $\sim 9\%$ but averages down with the number of contributing visits when evaluating the Fourier model at a science cell's $h_{\rm PDT}$.

### Diurnal phase interpretation

The pol-1 amplitude $A_1 = 13.2\ \mathrm{K}$ with phase $\phi_1 = +44^\circ$ corresponds to a maximum at $h_{\rm PDT} \approx 3.0$ h (before sunrise). The diode coupling depends on the LNA's input matching, which is sensitive to temperature: a colder LNA has a different reflection coefficient and the effective $T_{\rm cal}$ shifts. Receiver-hut temperature typically *minimises* near sunrise after a clear night, so a coincident *maximum* in $T_{\rm cal}$ is consistent with a negative temperature coefficient of the matching. The same physical mechanism plausibly drives the small pol-0 modulation; the second harmonic on pol 0 may reflect a thermal lag between outdoor temperature and the LNA, producing two daily inflexions.

### What this calibration does NOT cover

- **Elevation dependence**: the recal pointing's altitude varies between $20^\circ$ and $56^\circ$ over the diurnal cycle; ground spillover varies with altitude. The Fourier fit absorbs any correlated elevation-temperature effect, but a separate altitude term has not been fitted. The science map relies on this being negligible compared to the diurnal drift; this assumption could be tested by adding altitude as a second predictor.
- **Multi-week drift**: the dataset spans $\approx 4.5$ days; secular drift on weekly timescales (e.g.\ from outdoor seasonal trends) is not constrained. The science pipeline currently evaluates $T_{\rm cal}^{\rm pol1}(t)$ at the median cell time of each *session*; if science observations are taken weeks after the calibration campaign, a fresh recal_drift_bk run is warranted.
- **Pol-0 science use**: ruled out, as discussed above. Were the diode replaced with one that couples adequately at $3.2\ \mathrm{MHz}$ to pol 0, $\sqrt{2}$ SNR improvement on the science map would be straightforward.

### Handoff

The persisted state `artifacts/tcal_drift_state.pkl` contains the per-pol Fourier coefficients, the harmonic order, the $24$ h period, the PDT offset, and the recal pointing identifier. The science notebook `main_scan_data_products` loads it, evaluates $T_{\rm cal}^{\rm pol1}(t_{\rm cell})$ at each science cell's median time, and proceeds with the Y-factor formula to obtain $T_{\rm sys}^{\rm pol1}$ and finally $T_B = 2\,R^{\rm pol1}\,T_{\rm sys}^{\rm pol1}$.

### Conclusion

The EBHIS-anchored differential two-peak method gives a stable, physically motivated, time-resolved noise-diode calibration for pol 1 at the $\sim 9\%$ statistical and $\sim 3\%$ systematic level. The nominal manufacturer values are inadequate at $3.2\ \mathrm{MHz}$ sample rate; the diurnal drift recovered here is real ($30\sigma$ amplitude) and large enough that *not* correcting for it would bias the final $T_B$ map by $\sim 15\%$ peak-to-peak over the survey. The pol-0 calibration is not science-grade due to the diode coupling deficit and is excluded from the downstream Stokes-I recovery.

## Report figures (saved to `report/figures/`)


## Report figures (saved to `report/figures/`)


## Report figures (saved to `report/figures/`)


## Report figures (saved to `report/figures/`)


## Report figures (saved to `report/figures/`)


In [ ]:
# === Report figures (auto-saved to report/figures/) ===
from plotters import (savefig, plot_ebhis_vs_leuschner_R_per_pol,
                      plot_tcal_vs_time, plot_tcal_24h_fold)

fig, _ = plot_ebhis_vs_leuschner_R_per_pol(
    ebhis_spectra=ebhis_spectra,
    pointing_R_avg=POINTING_R_AVG,
    pointing_R_N=POINTING_R_N,
    ebhis_peak_v=EBHIS_PEAK_V, ebhis_peak_tb=EBHIS_PEAK_TB,
    R_peak_v=R_PEAK_V, R_peak_ref=R_PEAK_REF,
    v_lsr_axis=v_lsr_axis,
    pointing_labels=POINTING_LABELS,
    v_lsr_window=(V_LSR_LO, V_LSR_HI),
    n_visits_by_pointing=n_visits_by_pointing,
    pols=(1,),
)
savefig(fig, 'fig_ebhis_vs_R_pol1.pdf')

from ugradiolab.plotting import TEXTWIDTH_IN as _TW
fig, _ = plot_tcal_vs_time(tcal_df, POINTING_LABELS, pols=(1,),
                           width_in=_TW, height_ratio=0.30)
savefig(fig, 'fig_tcal_vs_time_pol1.pdf')

_design = lambda h, K: fourier_design(h, K, PERIOD_HOURS)
fig, _ = plot_tcal_24h_fold(
    fold, N_HARMONICS, _design,
    POINTING_LABELS[list(RECAL_POINTINGS.keys())[0]],
    pols=(1,), width_in=_TW, height_ratio=0.30,
)
savefig(fig, 'fig_tcal_24h_fold_pol1.pdf')
